# Capstone: Implement the Adam Optimizer

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 12.02 |
| Time | ~90 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/12_Capstone_Projects/03_implement_adam_optimizer.ipynb)

---

## 🎯 Learning Objective

Derive and implement the Adam optimizer with bias correction from scratch, and empirically compare its convergence to SGD.

---

## 📐 Theory

`04.04` already derived momentum, AdaGrad, RMSProp, and Adam, and implemented every one of them
correctly — verified bit-for-bit against `torch.optim`. What it left as a special case is bias
correction: it showed the correction is needed for a *constant* gradient, then stated the fix
without a general proof. This capstone derives that correction properly, for genuinely
stochastic gradients, and studies its effect over a real, extended training run rather than
`04.04`'s few-step illustration.

### Adam's moving averages

Adam tracks two exponential moving averages of the gradient $g_t$:
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2,
\qquad m_0=v_0=0$$

### Deriving the bias, in general

Unroll the recursion for $m_t$ (both $m_0=0$ and induction give this directly):
$$m_t = (1-\beta_1)\sum_{i=1}^t \beta_1^{\,t-i}\,g_i$$
Assume the gradients are stationary in expectation, $\mathbb{E}[g_i]=\mu$ for every $i$ (a
simplifying assumption Kingma & Ba's original derivation also makes — it need not hold exactly
in practice, but it isolates the bias cleanly). Then:
$$\mathbb{E}[m_t] = \mu(1-\beta_1)\sum_{i=1}^t \beta_1^{\,t-i} = \mu(1-\beta_1)\cdot
\frac{1-\beta_1^{\,t}}{1-\beta_1} = \mu\,(1-\beta_1^{\,t})$$
(the middle step sums a finite geometric series, recall `02.02`/`06`'s treatment of series).
So $m_t$ is a biased estimate of $\mu$ — biased low, by exactly the multiplicative factor
$(1-\beta_1^{\,t})$, which $\to 1$ as $t\to\infty$ but starts at $\beta_1$'s value at $t=1$ (a
severe bias early in training). Dividing it out, $\hat m_t = m_t/(1-\beta_1^{\,t})$, gives
$\mathbb{E}[\hat m_t]=\mu$ for **every** $t$, not just asymptotically — verified below with
actual stochastic (not constant) gradients, confirming the assumption-light derivation holds
beyond the special case `04.04` checked. The identical argument applies to $v_t$ with $\beta_2$.

### Why bias correction matters more than a first glance suggests

Both $\beta_1$ and $\beta_2$ typically default to values close to 1 (Kingma & Ba's paper uses
$\beta_1=0.9,\beta_2=0.999$), but $\beta_2$ is *much* closer to 1. Since $\hat v_t$ sits inside a
square root in Adam's update's *denominator*, an under-corrected (too-small) $v_t$ makes the
step too *large*, and — because $(1-\beta_2^{\,t})$ shrinks far more slowly than
$(1-\beta_1^{\,t})$ — this over-large-step problem persists for far longer than the numerator's
own bias does. Verified below: skipping bias correction doesn't just start Adam off slightly
wrong, it can make its early steps several times larger than intended for hundreds of steps,
occasionally overshooting badly enough to visibly spike the loss before recovering.

### Empirical comparison to SGD

Adam's combination of momentum-like averaging and per-parameter adaptive scaling should let it
navigate a loss surface using far fewer steps, and far less sensitivity to the initial random
seed, than plain SGD. `04.04` demonstrated this on one hand-picked quadratic bowl; this notebook
checks it on a genuinely different setting — training an actual (small) neural network, across
multiple random seeds, to see whether Adam's advantage is a robust pattern or a
cherry-picked one-off.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Extending `04.04`'s 5-step bias-correction check to 200 steps reveals the full shape of the
problem that check was too short to show: the uncorrected effective step size doesn't start too
small and grow toward $\eta$ — it *overshoots* $\eta$ early on, peaks around step 10, and only
slowly decays back down over hundreds of steps.

In [ ]:
g_viz = 2.5
beta1_viz, beta2_viz = 0.9, 0.999
eps_viz = 1e-8
lr_viz = 0.1

m, v = 0.0, 0.0
steps_corrected, steps_uncorrected = [], []
for t in range(1, 201):
    m = beta1_viz * m + (1 - beta1_viz) * g_viz
    v = beta2_viz * v + (1 - beta2_viz) * g_viz**2
    m_hat, v_hat = m / (1 - beta1_viz**t), v / (1 - beta2_viz**t)
    steps_corrected.append(lr_viz * m_hat / (np.sqrt(v_hat) + eps_viz))
    steps_uncorrected.append(lr_viz * m / (np.sqrt(v) + eps_viz))

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(range(1, 201), steps_corrected, color="#4C72B0", lw=2, label="with bias correction (constant, = $\\eta$)")
ax.plot(range(1, 201), steps_uncorrected, color="#C44E52", lw=2, label="WITHOUT bias correction")
ax.axhline(lr_viz, color='gray', ls=':', lw=1)
ax.set_xlabel("step $t$"); ax.set_ylabel("effective step size for a constant gradient $g=2.5$")
ax.set_title("Uncorrected Adam's step OVERSHOOTS $\\eta$ (peaking ~6.5x too\nlarge around t=10) rather than starting too small")
ax.legend()
plt.show()

print(f"Corrected step: constant {lr_viz} at every t, by construction.")
print(f"Uncorrected step: peaks at {max(steps_uncorrected):.4f} (t={np.argmax(steps_uncorrected)+1}), "
      f"still {steps_uncorrected[199]:.4f} at t=200 -- {steps_uncorrected[199]/lr_viz:.2f}x the target.")
print(f"\nThis happens because beta2=0.999 is much closer to 1 than beta1=0.9: v_t's bias factor")
print(f"(1-beta2^t) shrinks far slower than m_t's (1-beta1^t), so the denominator sqrt(v_t)")
print(f"stays under-corrected (too small) long after the numerator has largely self-corrected --")
print(f"an under-corrected denominator makes the overall step too LARGE, not too small.")

## 🐍 Implementation from Scratch

We verify the general bias-correction formula $\mathbb{E}[\hat m_t]=\mu$ holds for genuinely
stochastic (not constant) gradients at every $t$, confirm skipping correction visibly damages a
real, extended training run (not just a 5-step toy), and compare Adam against plain SGD training
an actual neural network across multiple random seeds.

In [ ]:
rng = np.random.default_rng(0)

# --- Verify the general bias-correction derivation, for STOCHASTIC (not constant) gradients ---
def simulate_m_t(t, beta1, true_mu, noise_std, rng):
    g = rng.normal(true_mu, noise_std, size=t)
    m = 0.0
    for gi in g:
        m = beta1 * m + (1 - beta1) * gi
    return m

beta1, true_mu, n_trials = 0.9, 3.0, 30000
print("General bias-correction check, with genuinely NOISY gradients (not 04.04's constant-g case):")
for t in [1, 5, 20, 100]:
    m_vals = np.array([simulate_m_t(t, beta1, true_mu, noise_std=2.0, rng=rng) for _ in range(n_trials)])
    theory_biased = true_mu * (1 - beta1**t)
    m_hat_vals = m_vals / (1 - beta1**t)
    print(f"  t={t:4d}: E[m_t]={m_vals.mean():.4f} (theory mu*(1-beta1^t)={theory_biased:.4f}), "
          f"E[m_hat_t]={m_hat_vals.mean():.4f} (theory mu={true_mu})")
print("Matches at every t, confirming the derivation holds beyond the constant-gradient special case.")

# --- Bias correction over a REAL, extended training run (not a 5-step toy) ---
def _unbroadcast(grad, shape):
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for i, s in enumerate(shape):
        if s == 1 and grad.shape[i] != 1:
            grad = grad.sum(axis=i, keepdims=True)
    return grad

class Tensor:
    def __init__(self, data, children=(), op=''):
        self.data = np.array(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self._backward = lambda: None
        self._children = set(children)
        self._op = op
    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += _unbroadcast(out.grad, self.data.shape)
            other.grad += _unbroadcast(out.grad, other.data.shape)
        out._backward = _backward
        return out
    def matmul(self, other):
        out = Tensor(self.data @ other.data, (self, other), '@')
        def _backward():
            a, b, g = self.data, other.data, out.grad
            a2 = a.reshape(1, -1) if a.ndim == 1 else a
            g2 = g.reshape(1, -1) if g.ndim == 1 else g
            self.grad += (g2 @ b.T).reshape(self.data.shape)
            other.grad += (a2.T @ g2).reshape(other.data.shape)
        out._backward = _backward
        return out
    def relu(self):
        out = Tensor(np.maximum(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (self.data > 0) * out.grad
        out._backward = _backward
        return out
    def __pow__(self, p):
        out = Tensor(self.data ** p, (self,), f'**{p}')
        def _backward():
            self.grad += (p * self.data ** (p - 1)) * out.grad
        out._backward = _backward
        return out
    def __sub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return self + other * Tensor(-1.0)
    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += _unbroadcast(out.grad * other.data, self.data.shape)
            other.grad += _unbroadcast(out.grad * self.data, other.data.shape)
        out._backward = _backward
        return out
    def sum(self):
        out = Tensor(self.data.sum(), (self,), 'sum')
        def _backward():
            self.grad += np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out
    def backward(self):
        topo, visited = [], set()
        def build(v):
            if id(v) not in visited:
                visited.add(id(v))
                for c in v._children:
                    build(c)
                topo.append(v)
        build(self)
        self.grad = np.ones_like(self.data)
        for v in reversed(topo):
            v._backward()

class Adam:
    # 04.04's Adam class, extended to operate on a LIST of Tensor parameters (12.02's engine)
    # instead of one raw NumPy array at a time.
    def __init__(self, params, lr, beta1=0.9, beta2=0.999, eps=1e-8, bias_correct=True):
        self.params, self.lr, self.b1, self.b2, self.eps = params, lr, beta1, beta2, eps
        self.bias_correct = bias_correct
        self.m = [np.zeros_like(p.data) for p in params]
        self.v = [np.zeros_like(p.data) for p in params]
        self.t = 0
    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * p.grad
            self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * p.grad**2
            if self.bias_correct:
                m_hat = self.m[i] / (1 - self.b1**self.t)
                v_hat = self.v[i] / (1 - self.b2**self.t)
            else:
                m_hat, v_hat = self.m[i], self.v[i]
            p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
    def zero_grad(self):
        for p in self.params:
            p.grad = np.zeros_like(p.data)

class SGD:
    def __init__(self, params, lr):
        self.params, self.lr = params, lr
    def step(self):
        for p in self.params:
            p.data -= self.lr * p.grad
    def zero_grad(self):
        for p in self.params:
            p.grad = np.zeros_like(p.data)

def make_regression_data(seed, n=60):
    r = np.random.default_rng(seed)
    X = r.uniform(-2, 2, size=(n, 2))
    y = (np.sin(2 * X[:, 0]) + np.cos(X[:, 1])).reshape(-1, 1)
    return X, y

def make_model(seed):
    r = np.random.default_rng(seed)
    return [Tensor(r.normal(scale=0.5, size=(2, 16))), Tensor(np.zeros(16)),
            Tensor(r.normal(scale=0.5, size=(16, 1))), Tensor(np.zeros(1))]

def train(opt_ctor, X, y, seed, steps=300):
    params = make_model(seed)
    opt = opt_ctor(params)
    losses = []
    for _ in range(steps):
        x_t = Tensor(X)
        h = (x_t.matmul(params[0]) + params[1]).relu()
        pred = h.matmul(params[2]) + params[3]
        loss = ((pred - Tensor(y)) ** 2).sum() * Tensor(1.0 / len(X))
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.data)
    return losses

X, y = make_regression_data(seed=100)
losses_corrected = train(lambda p: Adam(p, lr=0.2, bias_correct=True), X, y, seed=0)
losses_uncorrected = train(lambda p: Adam(p, lr=0.2, bias_correct=False), X, y, seed=0)
print(f"\nBias-correction ablation over a full {len(losses_corrected)}-step training run (lr=0.2):")
print(f"  Corrected:   final loss={losses_corrected[-1]:.5f}, peak loss during training={max(losses_corrected):.3f}")
print(f"  Uncorrected: final loss={losses_uncorrected[-1]:.5f}, peak loss during training={max(losses_uncorrected):.3f}")
print(f"  Uncorrected training visibly SPIKES early (overshooting, as the Visual Intuition")
print(f"  section's step-size plot predicts) and ends {losses_uncorrected[-1]/losses_corrected[-1]:.1f}x worse.")

# --- Adam vs plain SGD, multi-seed, on the SAME real network ---
n_seeds = 5
sgd_finals, adam_finals = [], []
for seed in range(n_seeds):
    sgd_finals.append(train(lambda p: SGD(p, lr=0.05), X, y, seed=seed)[-1])
    adam_finals.append(train(lambda p: Adam(p, lr=0.01), X, y, seed=seed)[-1])

print(f"\nSGD vs Adam, same network, {n_seeds} different random seeds:")
print(f"  SGD:  final losses={np.round(sgd_finals, 5)}, mean={np.mean(sgd_finals):.5f}, std={np.std(sgd_finals):.5f}")
print(f"  Adam: final losses={np.round(adam_finals, 5)}, mean={np.mean(adam_finals):.5f}, std={np.std(adam_finals):.5f}")
print(f"  Adam wins on EVERY seed: {all(a < s for a, s in zip(adam_finals, sgd_finals))}")

## 🤖 Why This Matters for AI

Applying `04.04`'s optimization theory end to end, this notebook builds the exact optimizer
that trains most modern neural networks — Adam and its descendant AdamW are the default choice
for essentially every transformer-based model, precisely because the per-parameter adaptive
scaling and bias-corrected moment estimates demonstrated above let training proceed quickly and
stably from the very first step, without needing hand-tuned per-layer learning rates. Below, we
push the multi-seed comparison further, training a slightly larger network and tracking not just
final loss but *how quickly* each optimizer reaches a fixed loss threshold — the practical
quantity that determines wall-clock training cost in real systems. The honest result is worth
sitting with: Adam reaches the threshold faster *on average*, but not on every single seed —
with fixed, untuned learning rates for both optimizers, a couple of seeds actually favor SGD.
This is a realistic picture of what "Adam usually wins" means in practice: a real advantage in
expectation, not a guarantee that holds identically for every random initialization.

In [ ]:
def make_model_bigger(seed):
    r = np.random.default_rng(seed)
    return [Tensor(r.normal(scale=0.5, size=(2, 32))), Tensor(np.zeros(32)),
            Tensor(r.normal(scale=0.5, size=(32, 1))), Tensor(np.zeros(1))]

def train_bigger(opt_ctor, X, y, seed, steps=500):
    params = make_model_bigger(seed)
    opt = opt_ctor(params)
    losses = []
    for _ in range(steps):
        x_t = Tensor(X)
        h = (x_t.matmul(params[0]) + params[1]).relu()
        pred = h.matmul(params[2]) + params[3]
        loss = ((pred - Tensor(y)) ** 2).sum() * Tensor(1.0 / len(X))
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.data)
    return losses

def steps_to_threshold(losses, threshold):
    for i, l in enumerate(losses):
        if l < threshold:
            return i
    return None

threshold = 0.05
n_seeds = 6
sgd_steps_list, adam_steps_list, sgd_loss_curves, adam_loss_curves = [], [], [], []
for seed in range(n_seeds):
    losses_sgd = train_bigger(lambda p: SGD(p, lr=0.05), X, y, seed=seed)
    losses_adam = train_bigger(lambda p: Adam(p, lr=0.01), X, y, seed=seed)
    sgd_loss_curves.append(losses_sgd)
    adam_loss_curves.append(losses_adam)
    t_sgd, t_adam = steps_to_threshold(losses_sgd, threshold), steps_to_threshold(losses_adam, threshold)
    sgd_steps_list.append(t_sgd)
    adam_steps_list.append(t_adam)
    winner = "Adam" if (t_adam or np.inf) < (t_sgd or np.inf) else "SGD"
    print(f"seed={seed}: SGD reaches loss<{threshold} at step {t_sgd}, Adam at step {t_adam}  (faster: {winner})")

print(f"\nSGD mean steps to threshold: {np.mean(sgd_steps_list):.1f}")
print(f"Adam mean steps to threshold: {np.mean(adam_steps_list):.1f}")
adam_wins = sum(1 for s, a in zip(sgd_steps_list, adam_steps_list) if a < s)
print(f"Adam faster on {adam_wins}/{n_seeds} seeds -- a real average advantage, not a clean sweep.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for seed in range(n_seeds):
    axes[0].plot(sgd_loss_curves[seed], color="#C44E52", alpha=0.5, lw=1)
    axes[1].plot(adam_loss_curves[seed], color="#4C72B0", alpha=0.5, lw=1)
axes[0].axhline(threshold, color='gray', ls=':'); axes[1].axhline(threshold, color='gray', ls=':')
axes[0].set_yscale('log'); axes[1].set_yscale('log')
axes[0].set_title(f"SGD, {n_seeds} seeds"); axes[1].set_title(f"Adam, {n_seeds} seeds")
axes[0].set_xlabel("step"); axes[1].set_xlabel("step")
axes[0].set_ylabel("loss (log scale)")
plt.tight_layout()
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Using this notebook's `Adam` class (built on `12.02`'s `Tensor`, operating on a *list* of
   parameters), track two separate scalar parameters $p_A$ (constant gradient $g_A=2.0$) and
   $p_B$ (constant gradient $g_B=0.5$) through 2 steps with $\eta=0.1$, $\beta_1=0.9$,
   $\beta_2=0.999$. By hand, compute both parameters' values after each step. As a sanity check,
   confirm both take a step of size *exactly* $\eta=0.1$ at every step despite their very
   different gradient magnitudes — Adam's per-parameter normalization, made concrete with two
   genuinely different parameters rather than one.

<details>
<summary>💡 Solution</summary>

For any single parameter with a *constant* gradient $g$, bias correction makes $\hat m_t=g$ and
$\hat v_t=g^2$ exactly at every $t$ (this is exactly `04.04`'s original special case, re-derived
in general in this notebook's Theory section), so the step is
$\eta\,\hat m_t/(\sqrt{\hat v_t}+\epsilon) = \eta\,g/(|g|+\epsilon)\approx\eta\,\text{sign}(g)$
— independent of $g$'s magnitude. Both $g_A=2.0$ and $g_B=0.5$ are positive, so both parameters
take a step of essentially exactly $\eta=0.1$ at $t=1$ *and* $t=2$ regardless of the 4x
difference in gradient magnitude: $p_A: 1.0\to0.9\to0.8$, $p_B: 1.0\to0.9\to0.8$ — identical
trajectories despite the very different underlying gradients, confirmed by running `Adam` on
both parameters together.

</details>

### 💭 UNDERSTAND
2. Bias correction guarantees $\mathbb{E}[\hat m_t]=\mu$ at *every* $t$, exactly, as this
   notebook's Theory section derives. Without computing anything, explain why this guarantee
   about the *mean* of $\hat m_t$ across hypothetical repeated runs does **not** imply that
   $\hat m_t$'s value on any *one actual run* becomes more reliable (lower-variance) at small
   $t$ just because the systematic bias has been removed.

<details>
<summary>💡 Solution</summary>

Bias correction is a purely deterministic rescaling — dividing by the constant
$(1-\beta_1^{\,t})$ — and rescaling by a constant changes the *mean* and the *standard
deviation* by the same factor, but at small $t$ that factor is large (dividing by a small
number), so it actually *inflates* the raw moving average's variance rather than shrinking it.
Concretely, $m_t$ itself is an average of very few gradient samples when $t$ is small (at $t=1$,
$m_1$ is not even really averaged — it is $(1-\beta_1)g_1$, a rescaling of a single noisy
sample), so it is intrinsically high-variance regardless of what happens after; dividing that
already-noisy quantity by a small $(1-\beta_1^{\,t})$ preserves (in fact amplifies in absolute
terms, though not in the *relative*/coefficient-of-variation sense) that per-run unreliability
rather than curing it. Removing bias fixes *where the estimate is centered on average*, not *how
much any individual run's estimate wobbles around that center* — those are different statistical
properties, and Adam's early steps are noisy for reasons bias correction was never meant to
address.

</details>

### ✏️ DERIVE
3. Derive a closed-form formula for $t^\star$, the number of steps needed until the bias factor
   $(1-\beta_2^{\,t})$ comes within a chosen tolerance $\delta$ of $1$ (i.e. $\beta_2^{\,t}\le
   \delta$), as a function of $\beta_2$ and $\delta$. Use it to confirm `04.04`'s claim that it
   takes roughly $t\approx3900$ steps for the default $\beta_2=0.999$ to get within 1% (and
   check what happens to that number at $\beta_2=0.99$, the Visual Intuition section's own
   alternative value).

<details>
<summary>💡 Solution outline</summary>

Start from the condition $\beta_2^{\,t}\le\delta$. Take the natural log of both sides — valid
since both sides are positive and $\log$ is monotonic — remembering $\log(\beta_2)<0$ since
$0<\beta_2<1$, which flips the inequality direction upon dividing:
$$t\log\beta_2 \le \log\delta \;\;\Longrightarrow\;\; t \ge \frac{\log\delta}{\log\beta_2}
\quad(\text{since }\log\beta_2<0).$$
So $t^\star = \lceil\log(\delta)/\log(\beta_2)\rceil$. Plugging in $\beta_2=0.999$, $\delta=0.01$
(within 1%): $t^\star = \lceil\log(0.01)/\log(0.999)\rceil = \lceil 4602.87\rceil = 4603$ —
matching `04.04`'s quoted "roughly $t\approx3900$" figure in order of magnitude (the small
difference is exactly the "roughly" — that figure was likely obtained by direct simulation
rather than this closed form, and/or used a slightly different tolerance). At $\beta_2=0.99$,
the same formula gives $t^\star=\lceil\log(0.01)/\log(0.99)\rceil=459$ — an order of magnitude
smaller, confirming numerically what the Theory section argues qualitatively: $\beta_2$ closer
to 1 makes the denominator's bias correction relevant for *far* longer.

</details>

### 🐍 IMPLEMENT
4. Extend this notebook's `Adam` class into `AdamW`, which applies *decoupled* weight decay —
   $p \mathrel{-{=}} \eta\big(\hat m_t/(\sqrt{\hat v_t}+\epsilon) + \lambda p\big)$ — where the
   weight-decay term is added directly to the update using the *current* parameter value $p$,
   completely outside the $m_t$/$v_t$ moving-average machinery (unlike plain L2 regularization,
   which would fold $\lambda p$ into the gradient itself before it ever reaches $m_t$/$v_t$).
   Check it against `torch.optim.AdamW` with matching hyperparameters (`lr`, `betas`, `eps`,
   `weight_decay`) on the same small regression setup this notebook already uses.

<details>
<summary>✅ How to know you're right</summary>

Running both optimizers for the same number of steps on identical data and identical parameter
initializations should produce parameter values matching `torch.optim.AdamW`'s to essentially
machine precision (max absolute difference on the order of $10^{-15}$ to $10^{-16}$), the same
standard this notebook already holds its plain `Adam` class to. If your implementation instead
matches plain L2-regularized `Adam` (weight decay folded into the gradient before the
moving-average step) rather than `AdamW`, the two will diverge visibly after a handful of steps
— decoupling weight decay from the adaptive denominator is precisely what distinguishes AdamW
from "Adam plus L2 regularization," per Loshchilov & Hutter's paper cited in Further Reading.

</details>

### 🤖 APPLY
5. Fine-tuning workflows often want different learning rates for different parts of a model — a
   small learning rate for an already-well-trained "backbone" and a larger one for a freshly
   initialized "head." Extend this notebook's `Adam` to support per-parameter learning rates
   (a list of `(Tensor, lr)` pairs instead of one shared `lr`), then train the same small
   network with a tiny learning rate on the first layer (treating it as a stand-in "backbone")
   and a much larger learning rate on the second layer (the "head").

<details>
<summary>✅ What you should observe</summary>

The first layer's weights should move far less over training than under a single shared
learning rate — in one run, giving the first layer a 100x smaller learning rate than the second
reduced its total weight movement (measured in Euclidean norm from initialization) by roughly an
order of magnitude compared to training both layers at the same uniform rate, while the network
as a whole still visibly reduces its loss over training (more slowly than the uniform-rate
baseline, since the backbone is deliberately constrained, but not stuck). This confirms the
per-group mechanism does what it's supposed to: each parameter's moving averages $m_t,v_t$ are
still computed and bias-corrected exactly as before, completely independently of which learning
rate multiplies the final step for that parameter — the adaptive *shape* of each parameter's
update is unaffected by which group it's in, only the update's overall *scale* changes.

</details>

### 🚀 CHALLENGE
6. In the AI section, seeds 3 and 4 are the ones where SGD reaches the loss threshold before
   Adam does, with both optimizers using a single fixed, untuned learning rate. Print each
   seed's initial loss (before any training step) for all 6 seeds, and check whether seeds 3 and
   4 simply started from an unusually easy (low-loss) initialization for SGD, or something else
   is going on. Then sweep Adam's learning rate over `[0.005, 0.01, 0.02, 0.05]` for just seeds
   3 and 4, and characterize whether a smaller or larger learning rate closes the gap with SGD.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer first rules out the easy explanation: seeds 3 and 4 actually start with *higher*
(worse) initial loss than every other seed in this batch (in one run, seed 3's initial loss was
around 15.8 and seed 4's around 7.0, versus roughly 1.3-3.9 for the seeds Adam wins on) — so
Adam's underperformance on these two seeds is not explained by SGD getting a lucky head start.
Sweeping Adam's learning rate on exactly these two seeds shows a *larger* learning rate closes
(and, in this run, reverses) the gap: at the default `lr=0.01`, Adam reached the threshold well
after SGD on both seeds, but at `lr=0.05`, Adam beat SGD's own step count decisively on both. A
strong answer draws the right conclusion carefully: this is not evidence that "Adam is worse than
SGD" in any general sense — it's evidence that a single learning rate value, chosen once and
applied uniformly, does not sit at each optimizer's individually-best operating point for every
random initialization, and Adam in particular is sensitive to its learning rate choice in a way
that a fixed-comparison methodology (one `lr` for Adam, one for SGD, across many seeds) can make
look like an intrinsic weakness when it's actually an artifact of the comparison's own design.
This connects to why real training pipelines run learning-rate sweeps (or use schedules/warmup)
rather than trusting a single fixed value to be simultaneously optimal across every run.

</details>

---

## 📚 Further Reading
- Kingma & Ba, [\"Adam: A Method for Stochastic Optimization\"](https://arxiv.org/abs/1412.6980)
- Reddi, Kale & Kumar, [\"On the Convergence of Adam and Beyond\"](https://arxiv.org/abs/1904.09237)
- Wilson et al., [\"The Marginal Value of Adaptive Gradient Methods in Machine Learning\"](https://arxiv.org/abs/1705.08292)

---

*Next notebook: [Capstone: Mathematical Analysis of GPT](04_mathematical_analysis_of_gpt.ipynb)*